## Setup
transfer_row_id uses monotonically_increasing_id(), not row_number() — deliberately different from the dimension surrogate keys. It only needs to be unique within this run to serve as a join key for the aggregation in Cell 7; it's never persisted as a stable identity, so there's no reason to pay row_number()'s single-partition cost here.


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "workspace"
SCHEMA = "transfermarkt"

def gold(t):
    return spark.read.parquet(f"/Volumes/{CATALOG}/{SCHEMA}/gold/{t}")

fact_transfer = gold("fact_transfer")
fact_appearance = gold("fact_appearance")
fact_player_valuation = gold("fact_player_valuation")
dim_player = gold("dim_player")
dim_club = gold("dim_club")
dim_date = gold("dim_date")

paid_transfers = (
    fact_transfer
    .filter(F.col("transfer_type") == "paid")
    .withColumn("transfer_row_id", F.monotonically_increasing_id())
)
print("Paid transfers:", paid_transfers.count())

## Target distribution and log transform

In [0]:
paid_transfers.select("transfer_fee").summary().show()

paid_transfers = paid_transfers.withColumn("log_transfer_fee", F.log1p("transfer_fee"))
paid_transfers.select("log_transfer_fee").summary().show()

## Coverage and fee inflation by year
Checks two things at once: whether fees have genuinely inflated over time (relevant to whether we need a temporal split, same reasoning as classification), and roughly where the appearances-based feature will start having data to work with.

In [0]:
transfer_years = (
    paid_transfers.join(dim_date, "date_key", "left")
    .groupBy("year")
    .agg(F.count("*").alias("transfer_count"),
         F.round(F.avg("transfer_fee"), 0).alias("avg_fee"),
         F.expr("percentile_approx(transfer_fee, 0.5)").alias("median_fee"))
    .orderBy("year")
)
transfer_years.show(30)

## Player age at transfer

In [0]:
transfers_with_date = paid_transfers.join(
    dim_date.select(F.col("date_key"), F.col("full_date").alias("transfer_full_date")),
    "date_key", "left"
)

transfers_with_player = transfers_with_date.join(
    dim_player.select("player_key", "date_of_birth", "position", "sub_position", "foot",
                       "height_in_cm", "international_caps", "international_goals"),
    "player_key", "left"
)

transfers_with_age = transfers_with_player.withColumn(
    "age_at_transfer", F.datediff("transfer_full_date", "date_of_birth") / 365.25
)
transfers_with_age.select("age_at_transfer").summary().show()

## Parsing total_market_value from text
The deferred task from Gold's Dim_Club cell — values like "€1.23bn" or "€45.00m" need to become real numbers now that this is a feature, not just a display field.

In [0]:
def parse_eur_shorthand(col):
    cleaned = F.regexp_replace(col, "[€,]", "")
    multiplier = (F.when(cleaned.rlike("bn$"), F.lit(1e9))
                    .when(cleaned.rlike("m$"), F.lit(1e6))
                    .when(cleaned.rlike("k$"), F.lit(1e3))
                    .otherwise(F.lit(1.0)))
    numeric_part = F.regexp_extract(cleaned, r"([0-9]*\.?[0-9]+)", 1).cast("double")
    return numeric_part * multiplier

dim_club_numeric = dim_club.withColumn("total_market_value_eur", parse_eur_shorthand(F.col("total_market_value")))

dim_club_numeric.filter(F.col("total_market_value").isNotNull()) \
    .select("total_market_value", "total_market_value_eur").show(10, truncate=False)

print("Parse failures (text present but numeric result null):",
      dim_club_numeric.filter(F.col("total_market_value").isNotNull() &
                                F.col("total_market_value_eur").isNull()).count())

## Club financial context
from_club/to_club value, plus the buying club's domestic competition as a league-quality proxy.

In [0]:
from_club = dim_club_numeric.select(F.col("club_key").alias("from_club_key"),
                                     F.col("total_market_value_eur").alias("from_club_value_eur"))
to_club = dim_club_numeric.select(F.col("club_key").alias("to_club_key"),
                                   F.col("total_market_value_eur").alias("to_club_value_eur"),
                                   F.col("domestic_competition_id").alias("to_domestic_competition_id"))

transfers_with_clubs = transfers_with_age.join(from_club, "from_club_key", "left").join(to_club, "to_club_key", "left")
transfers_with_clubs.select("from_club_value_eur", "to_club_value_eur").summary().show()

## Rolling 365-day performance — as-of range join
transfer_window is broadcast deliberately: it's tiny (17,554 rows) next to appearance_dates (1.89M), so broadcasting avoids shuffling the much larger table across the cluster just to join against a small one.

In [0]:
appearance_dates = fact_appearance.join(
    dim_date.select(F.col("date_key"), F.col("full_date").alias("appearance_full_date")), "date_key", "left"
).select("player_key", "appearance_full_date", "goals", "assists", "minutes_played")

transfer_window = transfers_with_clubs.select("transfer_row_id", "player_key", "transfer_full_date")

pre_transfer = appearance_dates.join(
    F.broadcast(transfer_window),
    (appearance_dates.player_key == transfer_window.player_key) &
    (appearance_dates.appearance_full_date < transfer_window.transfer_full_date) &
    (appearance_dates.appearance_full_date >= F.date_sub(transfer_window.transfer_full_date, 365)),
    "inner"
)

rolling_form = pre_transfer.groupBy("transfer_row_id").agg(
    F.sum("goals").alias("goals_last_365d"),
    F.sum("assists").alias("assists_last_365d"),
    F.sum("minutes_played").alias("minutes_last_365d"),
    F.count("*").alias("appearances_last_365d"),
)

transfers_final = transfers_with_clubs.join(rolling_form, "transfer_row_id", "left")
print("Transfers with no appearance history in the prior 365 days:",
      transfers_final.filter(F.col("appearances_last_365d").isNull()).count(), "out of", transfers_final.count())

## Assemble and write

In [0]:
cols = ["transfer_row_id", "transfer_fee", "log_transfer_fee", "date_key",
        "player_key", "age_at_transfer", "position", "sub_position", "foot", "height_in_cm",
        "international_caps", "international_goals", "market_value_in_eur",
        "from_club_value_eur", "to_club_value_eur", "to_domestic_competition_id",
        "goals_last_365d", "assists_last_365d", "minutes_last_365d", "appearances_last_365d"]

regression_features = transfers_final.select(*cols)
regression_features.write.mode("overwrite").parquet(f"/Volumes/{CATALOG}/{SCHEMA}/ml/regression_features")
print("Total rows:", regression_features.count())
regression_features.show(5)

## Club value, done as-of the transfer date
Computed once per distinct transfer date rather than once per transfer row, since many transfers share deadline-day dates. Cached because it gets joined twice below (from_club and to_club) — without caching, Spark would recompute this expensive lookup from scratch for each join.

In [0]:
reference_dates = transfers_final.select(F.col("transfer_full_date").alias("ref_date")).distinct()
print("Distinct transfer dates needing a club-value snapshot:", reference_dates.count())

valuations_with_date = fact_player_valuation.join(
    dim_date.select(F.col("date_key"), F.col("full_date").alias("valuation_full_date")), "date_key", "left"
).select("player_key", "valuation_full_date", "club_key", "market_value_in_eur")

as_of_valuations = valuations_with_date.join(
    F.broadcast(reference_dates),
    valuations_with_date.valuation_full_date <= reference_dates.ref_date,
    "inner"
)

latest_val_window = Window.partitionBy("player_key", "ref_date").orderBy(F.desc("valuation_full_date"))

club_value_as_of = (
    player_club_as_of.groupBy("ref_date", "club_key")
    .agg(F.sum("market_value_in_eur").alias("club_squad_value_eur"))
)

club_value_as_of.write.mode("overwrite").parquet(f"/Volumes/{CATALOG}/{SCHEMA}/ml/_tmp_club_value_as_of")
club_value_as_of = spark.read.parquet(f"/Volumes/{CATALOG}/{SCHEMA}/ml/_tmp_club_value_as_of")

print("club_value_as_of rows:", club_value_as_of.count())

## Join club value onto transfers

In [0]:
from_club_value = club_value_as_of.select(
    F.col("ref_date").alias("from_ref_date"),
    F.col("club_key").alias("from_club_key_lookup"),
    F.col("club_squad_value_eur").alias("from_club_value_eur"))
to_club_value = club_value_as_of.select(
    F.col("ref_date").alias("to_ref_date"),
    F.col("club_key").alias("to_club_key_lookup"),
    F.col("club_squad_value_eur").alias("to_club_value_eur"))

transfers_with_club_value = (
    transfers_final
    .drop("from_club_value_eur", "to_club_value_eur")
    .join(from_club_value,
          (transfers_final.transfer_full_date == from_club_value.from_ref_date) &
          (transfers_final.from_club_key == from_club_value.from_club_key_lookup), "left")
    .join(to_club_value,
          (transfers_final.transfer_full_date == to_club_value.to_ref_date) &
          (transfers_final.to_club_key == to_club_value.to_club_key_lookup), "left")
    .drop("from_ref_date", "to_ref_date", "from_club_key_lookup", "to_club_key_lookup")
)

transfers_with_club_value.select("from_club_value_eur", "to_club_value_eur").summary().show()
print("Missing from_club_value:", transfers_with_club_value.filter(F.col("from_club_value_eur").isNull()).count())
print("Missing to_club_value:", transfers_with_club_value.filter(F.col("to_club_value_eur").isNull()).count())

## History flag, age floor, and final scoping
has_recent_appearance_history preserves the "unknown" vs "genuinely zero" distinction instead of collapsing it. Age floor of 16 drops the small youth-transfer population that doesn't represent the same phenomenon as adult transfer pricing. Year window (2010-2026) drops the unstable, tiny early years and the single stray 2027 row.

In [0]:
transfers_with_flags = (
    transfers_with_club_value
    .withColumn("has_recent_appearance_history", F.col("appearances_last_365d").isNotNull())
    .withColumn("goals_last_365d", F.coalesce("goals_last_365d", F.lit(0)))
    .withColumn("assists_last_365d", F.coalesce("assists_last_365d", F.lit(0)))
    .withColumn("minutes_last_365d", F.coalesce("minutes_last_365d", F.lit(0)))
    .withColumn("appearances_last_365d", F.coalesce("appearances_last_365d", F.lit(0)))
)

transfers_scoped = (
    transfers_with_flags
    .join(dim_date.select(F.col("date_key"), F.col("year")), "date_key", "left")
    .filter((F.col("age_at_transfer") >= 16) & (F.col("year") >= 2010) & (F.col("year") <= 2026))
)

print("Rows after age floor + 2010-2026 scoping:", transfers_scoped.count())
transfers_scoped.groupBy("has_recent_appearance_history").count().show()
transfers_scoped.select("from_club_value_eur", "to_club_value_eur").summary().show()

transfers_scoped.write.mode("overwrite").parquet(f"/Volumes/{CATALOG}/{SCHEMA}/ml/regression_features_final")
print("Final row count:", transfers_scoped.count())

## What the finished feature table actually looks like

In [0]:
regression_features_final = spark.read.parquet(f"/Volumes/{CATALOG}/{SCHEMA}/ml/regression_features_final")
regression_features_final.printSchema()
regression_features_final.show(10, truncate=False)

## Log-transform the money features for these checks
Same transform Cell 2 of the model notebook will apply later. Doing it here too isn't duplicated work so much as it's the honest way to run this check — correlation/linearity against the raw skewed euro values would be checking the wrong thing.

In [0]:
regression_features_final = (
    regression_features_final
    .withColumn("log_market_value", F.log1p("market_value_in_eur"))
    .withColumn("log_from_club_value", F.log1p("from_club_value_eur"))
    .withColumn("log_to_club_value", F.log1p("to_club_value_eur"))
)

## Linearity check: correlation of each predictor with the target
Pearson correlation only measures linear association specifically — a real, strong, but curved relationship (age is the classic case, given transfer fees typically peak in a player's mid-20s and decline afterward) can show up as a weak correlation here despite genuinely mattering. That's exactly why this gets paired with the scatter plots below rather than used alone.

In [0]:
continuous_cols_to_check = ["age_at_transfer", "height_in_cm", "international_caps", "international_goals",
                             "log_market_value", "log_from_club_value", "log_to_club_value",
                             "goals_last_365d", "assists_last_365d", "minutes_last_365d", "appearances_last_365d"]

for c in continuous_cols_to_check:
    corr = regression_features_final.stat.corr(c, "log_transfer_fee")
    print(f"{c}: {corr:.3f}")

## Multicollinearity check among predictors
This matters for a specific reason tied to why Linear Regression was chosen at all: we picked it partly for interpretable coefficients ("each extra goal is worth X% more fee"). If two predictors are highly correlated with each other, the model can arbitrarily split credit between them, making individual coefficients unstable even if the model's overall predictions are fine. If this turns up meaningful pairs, that's not a reason to abandon the approach — it's confirmation that the ElasticNet regularization already planned for Cell 6 of the model notebook is doing real work, not just a tuning formality.

In [0]:
import itertools

for c1, c2 in itertools.combinations(continuous_cols_to_check, 2):
    corr = regression_features_final.stat.corr(c1, c2)
    if abs(corr) > 0.6:
        print(f"{c1} vs {c2}: {corr:.3f}")

## Scatter plots against the target
17,218 rows is small enough to pull to the driver directly — no need to subsample. alpha=0.15 handles overplotting so density patterns are visible instead of a solid blob. What to look for: a roughly linear cloud (even if wide/noisy) is a good sign; an obvious curve suggests a transform or an extra term is needed; a fan/funnel shape (spread widening or narrowing across the x-range) is a heteroscedasticity warning sign, though that's confirmed properly with residuals later, not from this alone.

In [0]:
import matplotlib.pyplot as plt

pdf = regression_features_final.select(
    "log_transfer_fee", "log_market_value", "age_at_transfer",
    "goals_last_365d", "log_from_club_value", "log_to_club_value"
).toPandas()

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
features_to_plot = ["log_market_value", "age_at_transfer", "goals_last_365d",
                     "log_from_club_value", "log_to_club_value"]

for ax, feat in zip(axes.flat, features_to_plot):
    ax.scatter(pdf[feat], pdf["log_transfer_fee"], alpha=0.15, s=8)
    ax.set_xlabel(feat)
    ax.set_ylabel("log_transfer_fee")

axes.flat[-1].axis("off")
plt.tight_layout()
plt.show()